# End-to-End Probe Inference Demo

This notebook walks through the Evee pathogenicity prediction pipeline:
1. Load pre-extracted Evo2 activations for 8 ClinVar variants
2. Inspect the activation tensor layout
3. Run the bilinear covariance probe with Newton-Schulz spectral compression
4. Compare predictions to ground truth

**No GPU required.** All artifacts are pre-computed and included in the repo.
Only standard libraries: `torch`, `safetensors`, `polars`.

In [ ]:
from pathlib import Path

import polars as pl
import safetensors.torch
import torch
from torch import Tensor

SAMPLES = Path("../artifacts/samples")

## 1. Load Sample Variants

8 curated ClinVar variants from the gene-holdout test set:
4 pathogenic + 4 benign, across missense, splice, intronic, and synonymous.

In [ ]:
variants = pl.read_ipc(SAMPLES / "variants.feather")
variants

## 2. Load Activations

Each variant has its own safetensors file containing a `[2, 2, 256, 4096]` tensor:

| Dim | Size | Meaning |
|-----|------|---------|
| 0   | 2    | Direction: forward, backward (reverse complement) |
| 1   | 2    | View: variant sequence, reference sequence |
| 2   | 256  | Positions selected by cosine divergence |
| 3   | 4096 | Evo2-7B block 27 hidden dimension |

The 256 positions are where the model's internal representation differs most
between variant and reference — where it "notices" the mutation.

In [ ]:
variant_ids = variants["variant_id"].to_list()
activations = torch.stack([
    safetensors.torch.load_file(str(SAMPLES / f"{vid.replace(':', '_')}.safetensors"))["activations"]
    for vid in variant_ids
]).float()

print(f"Loaded {activations.shape[0]} variants: {activations.shape}")

In [ ]:
# Inspect the first variant: COL1A1 missense pathogenic
var_fwd = activations[0, 0, 0]   # variant, forward direction
ref_fwd = activations[0, 0, 1]   # reference, forward direction
diff = var_fwd - ref_fwd

print(f"Variant: {variant_ids[0]} ({variants[0, 'gene_name']}, {variants[0, 'consequence']})")
print(f"  Variant activation norm:  {var_fwd.norm(dim=-1).mean():.2f}")
print(f"  Reference activation norm: {ref_fwd.norm(dim=-1).mean():.2f}")
print(f"  Difference norm:           {diff.norm(dim=-1).mean():.4f}")

## 3. Bilinear Covariance Probe

The probe architecture has three stages:

1. **Untied projection**: Two separate linear maps (left, right) per direction
   project each position from 4096-d to 64-d
2. **Covariance pooling**: `left.T @ right / K` captures second-order
   interactions across all 256 positions → a 64×64 matrix per direction.
   Diagonal regularization (`εI`) ensures stability, then **Newton-Schulz
   spectral compression** (matrix square root) dampens dominant eigenvalues.
3. **Bilinear readout**: A factored bilinear form on the compressed
   covariance produces pathogenicity logits.

In [ ]:
def newton_schulz_sqrtm(matrix: Tensor, n_iters: int = 3) -> Tensor:
    """Matrix square root via Newton-Schulz coupled iteration.

    Normalizes by Frobenius norm to bound eigenvalues to (0, 1],
    runs the iteration, then rescales. Upcasts to float32 for stability.
    """
    dtype = matrix.dtype
    m = matrix.float()
    d = m.size(-1)

    frob = m.flatten(-2).norm(dim=-1)[..., None, None]
    scale = frob.sqrt()
    normed = m / frob

    eye = torch.eye(d, device=m.device)
    y, z = normed, eye.expand_as(m).clone()
    for _ in range(n_iters):
        t = 3 * eye - z @ y
        y = y @ t / 2
        z = t @ z / 2

    return (y * scale).to(dtype)


class CovarianceProbe:
    """Bilinear covariance probe for bidirectional activations.

    Loads weights from a safetensors file and runs inference on
    pre-extracted Evo2 activations.

    Architecture::

        diff_d = var_same_d - ref_same_d          (per direction d)
        left_d = proj_left_d(diff_d)              [B, K, 64]
        right_d = proj_right_d(diff_d)            [B, K, 64]
        cov = Σ_d left_d.T @ right_d / K + εI    [B, 64, 64]
        cov = sqrtm(cov)                          Newton-Schulz spectral compression
        feat = W_left @ cov @ W_right.T           [B, 128] (bilinear readout)
        logits = out(feat)                        [B, 2]
    """

    def __init__(self, weights: dict[str, Tensor], eps: float = 1e-3, n_sqrtm_iters: int = 3):
        self.w = weights
        self.eps = eps
        self.n_sqrtm_iters = n_sqrtm_iters

    @classmethod
    def load(cls, path: Path, **kwargs) -> "CovarianceProbe":
        return cls(safetensors.torch.load_file(str(path)), **kwargs)

    @torch.no_grad()
    def __call__(self, activations: Tensor) -> Tensor:
        """[B, 2, 2, K, d] → [B] pathogenicity scores."""
        w = self.w

        # Variant − reference diff per direction
        diff_fwd = activations[:, 0, 0] - activations[:, 0, 1]
        diff_bwd = activations[:, 1, 0] - activations[:, 1, 1]

        # Project to low-dim (separate left/right per direction)
        left_fwd = diff_fwd @ w["proj_left_fwd"].T + w["proj_left_fwd_bias"]
        right_fwd = diff_fwd @ w["proj_right_fwd"].T + w["proj_right_fwd_bias"]
        left_bwd = diff_bwd @ w["proj_left_bwd"].T + w["proj_left_bwd_bias"]
        right_bwd = diff_bwd @ w["proj_right_bwd"].T + w["proj_right_bwd_bias"]

        # Covariance pooling + regularization
        K = left_fwd.shape[1]
        cov = (
            torch.bmm(left_fwd.transpose(1, 2), right_fwd) / K
            + torch.bmm(left_bwd.transpose(1, 2), right_bwd) / K
            + self.eps * w["eye"].unsqueeze(0)
        )

        # Newton-Schulz spectral compression
        cov = newton_schulz_sqrtm(cov, self.n_sqrtm_iters)

        # Bilinear readout
        feat = torch.einsum("blr,hl,hr->bh", cov, w["head_left"], w["head_right"])
        logits = feat @ w["out_weight"].T + w["out_bias"]
        return torch.softmax(logits, dim=-1)[:, 1]

In [ ]:
probe = CovarianceProbe.load(SAMPLES / "probe.safetensors")
print("Probe weights:")
for name, tensor in probe.w.items():
    print(f"  {name}: {tensor.shape}")

## 4. Run Inference

In [ ]:
scores = probe(activations)

results = variants.with_columns(
    pl.Series("score", [round(s, 4) for s in scores.tolist()]),
    pl.Series("prediction", ["pathogenic" if s > 0.5 else "benign" for s in scores.tolist()]),
).with_columns(
    (pl.col("prediction") == pl.col("label")).alias("correct"),
)
results.select("variant_id", "gene_name", "consequence", "label", "score", "prediction", "correct")

In [ ]:
n_correct = results["correct"].sum()
print(f"Accuracy: {n_correct}/{results.height} ({100 * n_correct / results.height:.0f}%)")

## Summary

The pipeline extracts Evo2-7B block 27 activations at 256 divergent
positions per variant, computes the variant−reference diff from both
reading directions, pools via cross-covariance with Newton-Schulz
spectral compression, and classifies with a bilinear readout.

The full model achieves **0.97 AUROC** on 34K held-out variants
(gene-level split), outperforming CADD, AlphaMissense, and other
computational predictors across all consequence types.